# Multi-Output GP

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
from datasets import load_dataset
from data_processing import process_one_obj_one_band, process_one_obj_one_band_train_heldout, split_train_heldout_observations, select_examples_and_process_train_heldout
from MOGP_model import (
    run_mogp_evaluation
)
from visualization import plot_gp_fit, plot_largest_standardized_residual_cases

dset_plasticc = load_dataset("MultimodalUniverse/plasticc",
                       streaming=True,
                       split='train')
dset_plasticc = dset_plasticc.with_format("numpy")

/Users/green/Downloads/multi_outputGP/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
mogp_params = {
    "heldout_fraction": 0.2,
    "min_train_points": 5,
    "scale_mode": "local_peak",
    "time_length_scale": 0.3,
    "wavelength_length_scale": 200.0,
    "n_restarts_optimizer": 0,
}

result = run_mogp_evaluation(
    dset_plasticc,
    n_objects=10,
    **mogp_params,
)

total train: 76
total heldout: 22
MOGP learned kernel: 0.373**2 * Matern(length_scale=[0.33, 20], nu=1.5)
total train: 73
total heldout: 22
MOGP learned kernel: 0.438**2 * Matern(length_scale=[2.25, 2e+03], nu=1.5)
total train: 79
total heldout: 22
MOGP learned kernel: 0.323**2 * Matern(length_scale=[0.487, 498], nu=1.5)
total train: 79
total heldout: 22
total train: 79
total heldout: 23
total train: 81
total heldout: 22


/Users/green/Downloads/multi_outputGP/.venv/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified lower bound 20.0. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/green/Downloads/multi_outputGP/.venv/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified upper bound 2000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/green/Downloads/multi_outputGP/.venv/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(

total train: 78
total heldout: 22
total train: 76
total heldout: 21
total train: 74
total heldout: 22
total train: 81
total heldout: 23


In [4]:
import pandas as pd
import numpy as np

rows = []
for m in result["object_results"]:
    z = (m["y_true_raw"] - m["y_pred_raw"]) / np.maximum(m["y_std_raw"], 1e-12)

    for i in range(m["n_heldout"]):
        rows.append({
            "object_id": m["object_id"][i],
            "band": m["band"][i],
            "time": m["time"][i],
            "y_true": m["y_true_raw"][i],
            "y_pred": m["y_pred_raw"][i],
            "y_std": m["y_std_raw"][i],
            "z": z[i],
            "abs_z": abs(z[i]),
        })

df = pd.DataFrame(rows)
df.sort_values("abs_z", ascending=False).head(20)

,object_id,band,time,y_true,y_pred,y_std,z,abs_z
14,21846282,r,0.065589,40.877480,72.772604,8.204328,-3.887597,3.887597
43,114421739,i,2.656343,-18.363379,1.050943,6.906805,-2.810898,2.810898
142,16775731,z,-0.049775,-22.325037,39.631278,22.248975,-2.784682,2.784682
8,21846282,r,-0.299802,0.951507,18.877242,7.578400,-2.365372,2.365372
33,114421739,r,1.512123,2.520881,14.638953,5.156114,-2.350234,2.350234
144,16775731,u,0.023137,14.030240,50.351505,16.155480,-2.248232,2.248232
11,21846282,z,-0.166034,-8.066343,19.159370,12.589182,-2.162628,2.162628
167,31971611,z,0.164612,-0.418611,30.364599,15.687420,-1.962286,1.962286
38,114421739,z,2.448782,-20.877531,4.411828,13.299893,-1.901471,1.901471
196,50875482,i,2.729796,7.258542,-1.193511,4.569158,1.849805,1.849805


In [5]:
print(result["aggregate"]["observation_weighted"])
print(result["aggregate"]["object_weighted"])

{'nlpd': 3.924748409314256, 'rmse': 25.385498257359902, 'coverage_1sigma': 0.751131221719457, 'coverage_2sigma': 0.9683257918552036, 'coverage_3sigma': 0.995475113122172}
{'nlpd': 3.9217871861116853, 'rmse': 18.216961200555744, 'coverage_1sigma': 0.7522962544701675, 'coverage_2sigma': 0.9681818181818181, 'coverage_3sigma': 0.9954545454545455}
